In [5]:
import pandas as pd
import numpy as np
import optuna
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Load train and test
train_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded2.csv")
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded2.csv")

In [3]:
# ----------------------
# 1. Prepare Data
# ----------------------
# Replace 'target' with actual target column
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

y_train = y_train.astype('category')
y_test = y_test.astype('category')

In [4]:
label_mapping = {-1: 0, 0: 1, 1: 2}

y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)

In [6]:
# ----------------------
# 2. Feature Selection using Random Forest
# ----------------------
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False)

# Select top N features (e.g., top 21)
selected_features = top_features.head(21).index.tolist()
selected_features

['order_performance_score',
 'order_to_shipment_planned_days',
 'shipping_mode',
 'customer_performance_score',
 'distance_normalized',
 'order_item_discount',
 'order_profit_per_order',
 'profit_per_order',
 'order_item_profit_ratio',
 'sales',
 'shipment_delay_days',
 'order_item_product_price',
 'shipping_hour_cos',
 'order_hour_cos',
 'shipping_hour_sin',
 'order_hour_sin',
 'shipping_dayofweek_sin',
 'order_shipping_time',
 'order_dayofweek_sin',
 'order_item_quantity',
 'order_to_shipment_days']

In [7]:
X_train_reduced = X_train[selected_features]
X_test_reduced = X_test[selected_features]

In [8]:
# ----------------------
# 2. Optuna Objective for LightGBM
# ----------------------
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', -1, 15),  # -1 means no limit
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'random_state': 42,
        'n_jobs': -1,
        'objective': 'multiclass',
        'num_class': len(np.unique(y_train)),
        'metric': 'multi_logloss'
    }
    
    model = LGBMClassifier(**params)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    score = cross_val_score(
        model,
        X_train_reduced,
        y_train,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1
    )
    
    return score.mean()

In [10]:
# ----------------------
# 3. Run Optimization
# ----------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200, show_progress_bar=True)

print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

[I 2025-08-12 17:02:59,907] A new study created in memory with name: no-name-0a97e847-6dc1-47b6-9d52-54443dbcee04
Best trial: 0. Best value: 0.724373:   0%|          | 1/200 [00:09<30:20,  9.15s/it]

[I 2025-08-12 17:03:09,063] Trial 0 finished with value: 0.7243726719059488 and parameters: {'n_estimators': 152, 'max_depth': 10, 'learning_rate': 0.075414816093591, 'num_leaves': 80, 'subsample': 0.6335172744824666, 'colsample_bytree': 0.8079816312118986, 'min_child_samples': 88, 'reg_alpha': 2.9681952595252996, 'reg_lambda': 2.8641690606266916}. Best is trial 0 with value: 0.7243726719059488.


Best trial: 0. Best value: 0.724373:   1%|          | 2/200 [00:15<25:42,  7.79s/it]

[I 2025-08-12 17:03:15,893] Trial 1 finished with value: 0.6764520840966631 and parameters: {'n_estimators': 111, 'max_depth': 6, 'learning_rate': 0.016037865726933748, 'num_leaves': 104, 'subsample': 0.9691352770909745, 'colsample_bytree': 0.7769517413139169, 'min_child_samples': 49, 'reg_alpha': 3.429958057944641, 'reg_lambda': 2.7401990231091795}. Best is trial 0 with value: 0.7243726719059488.


Best trial: 2. Best value: 0.743776:   2%|▏         | 3/200 [01:17<1:45:23, 32.10s/it]

...

Best trial: 98. Best value: 0.75892:  51%|█████     | 102/200 [4:31:51<16:08:56, 593.23s/it]

[I 2025-08-12 21:34:51,317] Trial 101 finished with value: 0.7560728572054795 and parameters: {'n_estimators': 941, 'max_depth': 14, 'learning_rate': 0.1287097255821389, 'num_leaves': 206, 'subsample': 0.6076599798750693, 'colsample_bytree': 0.6317373966460129, 'min_child_samples': 5, 'reg_alpha': 0.04720792411771548, 'reg_lambda': 4.806462809738799}. Best is trial 98 with value: 0.7589203190343639.


Best trial: 98. Best value: 0.75892:  51%|█████     | 102/200 [4:31:53<4:21:13, 159.93s/it] 


[W 2025-08-12 21:34:53,216] Trial 102 failed with parameters: {'n_estimators': 976, 'max_depth': 14, 'learning_rate': 0.1279508332782794, 'num_leaves': 206, 'subsample': 0.6014717503129988, 'colsample_bytree': 0.631608750547893, 'min_child_samples': 5, 'reg_alpha': 0.04564944108287671, 'reg_lambda': 4.831080937501918} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\VP678WV\OneDrive - EY\Documents\Delivery_Delay\logistic_venv\lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\VP678WV\AppData\Local\Temp\ipykernel_33896\3922019691.py", line 26, in objective
    score = cross_val_score(
  File "c:\Users\VP678WV\OneDrive - EY\Documents\Delivery_Delay\logistic_venv\lib\site-packages\sklearn\utils\_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\VP678WV\OneDrive - EY\Documents\Delivery_Delay\logistic_venv\lib\site-packages\

KeyboardInterrupt: 

In [11]:
best_params = {'n_estimators': 991, 'max_depth': 15, 'learning_rate': 0.11493042829241355, 'num_leaves': 218, 'subsample': 0.571672074678408, 'colsample_bytree': 0.6282051583603918, 'min_child_samples': 6, 'reg_alpha': 0.0020533664937874008, 'reg_lambda': 4.850360590699926}

In [13]:
# ----------------------
# 4. Train Final Model
# ----------------------
# best_params = study.best_params
best_params.update({
    'random_state': 42,
    'n_jobs': -1,
    'objective': 'multiclass',
    'num_class': len(np.unique(y_train)),
    'metric': 'multi_logloss'
})

lgb_best = LGBMClassifier(**best_params)
lgb_best.fit(X_train_reduced, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002332 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2910
[LightGBM] [Info] Number of data points in the train set: 15799, number of used features: 21
[LightGBM] [Info] Start training from score -1.150509
[LightGBM] [Info] Start training from score -1.248563
[LightGBM] [Info] Start training from score -0.924808
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

LGBMClassifier(colsample_bytree=0.6282051583603918,
               learning_rate=0.11493042829241355, max_depth=15,
               metric='multi_logloss', min_child_samples=6, n_estimators=991,
               n_jobs=-1, num_class=3, num_leaves=218, objective='multiclass',
               random_state=42, reg_alpha=0.0020533664937874008,
               reg_lambda=4.850360590699926, subsample=0.571672074678408)

In [14]:
# ----------------------
# 5. Test Evaluation
# ----------------------
y_pred = lgb_best.predict(X_test_reduced)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Macro:", f1_score(y_test, y_pred, average='macro'))

Test Accuracy: 0.6109324758842444
Test F1 Macro: 0.5475578814277321
